In [1]:
# https://light-tree.tistory.com/196
# PCGrad

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

# !cp "/content/drive/MyDrive/retrieval-contrastive/book_meta.parquet" "/content"

In [3]:
# import gdown

# url = "https://drive.google.com/file/d/1rmKguIJREhMcTkZyM3arAXUX3exQZG2W/view?usp=sharing"
# gdown.download(url, output="book_meta.parquet", quiet=False)

**Teacher 모델**: 원본 E5
  * Output: `(B, L, D)` → token-level hidden states
  * 문장 embedding 얻으려면 token-level pooling 필요 → mask 꼭 사용

**Student 모델**: LoRA만 붙인 E5, MultiVectorHead 없음
  * Output: `(B, L, D)` → token-level hidden states
  * 문장 embedding 얻으려면 teacher처럼 **token-level pooling + mask** 사용
  * MultiVectorHead 없으면 student는 K개 vector가 아니라 단일 sentence vector

**Student 모델**: LoRA + MultiVectorHead 있는 경우
  * Output: `(B, K, D)` → 이미 pooled된 multi-vector
  * **pooling 하면 안 됨**, mask 필요 없음

| 모델                               | Output shape | Pooling 필요 | Mask 필요 |
| -------------------------------- | ------------ | ---------- | ------- |
| Teacher                          | (B, L, D)    | ✅ yes      | ✅ yes   |
| Student (LoRA only)              | (B, L, D)    | ✅ yes      | ✅ yes   |
| Student (LoRA + MultiVectorHead) | (B, K, D)    | ❌ no       | ❌ no    |

In [4]:
# Add-on Layer: 카테고리 임베딩 레이어(Category Embedding)와 이 둘을 합치는 퓨전 레이어(Linear Layer)만 새로 만듭니다.
# 이렇게 모델 구조를 뜯어고치는 건 나중에 '평점'이나 '페이지 수' 같은 숫자를 꼭 넣어야 할 때 하셔도 늦지 않습니다.
import random
import math
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data import random_split

from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from transformers import get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType #, AdapterConfig

In [5]:
def set_seed(seed: int = 42):
    random.seed(seed)            # 기본 Python random 고정
    np.random.seed(seed)         # NumPy 랜덤 고정
    torch.manual_seed(seed)      # CPU 연산 랜덤 고정
    torch.cuda.manual_seed(seed) # GPU 모든 디바이스 랜덤 고정
    torch.cuda.manual_seed_all(seed)  # 멀티 GPU일 때

    # 연산 재현성
    torch.backends.cudnn.deterministic = True  # cuDNN 연산을 determinisitc으로 강제
    torch.backends.cudnn.benchmark = False     # CUDA 성능 자동 튜닝 기능 끔 → 완전 재현 가능

set_seed(42)

In [6]:
EPOCHS = 20
WARMUP_RATIO = 0.1 # warmup = 학습 초반에 LR을 낮게 시작해서 천천히 올리는 기법 / Transformer, BERT, GPT 등에서 매우 중요

너의 contrastive loss 구조는:

positive 끼리 한 점처럼 뭉치기 / negative에서 멀어지기
+ 온도(temperature)=0.05 같은 작은 값
→ exp scaling 때문에 gradient 초강함.. 그럼 cluster collapse 되는 경향이 생김
즉, retrieval에는 오히려 불리한 구조.

retrieval은 “카테고리별 한 점에 뭉치기”가 아니라
semantic 분산 유지 + category 분리의 균형을 원하거든.
    
contrastive가 너무 강력해서 embedding이 collapse되므로,
강한 distillation(λ=200+)이 오히려 retrieval 성능을 높였다는 것은 100% 합리적이다.

In [7]:
# # 일단 이건 128 배치에서 최적값인듯
LEARNING_RATE = 1e-3
BATCH_SIZE = 128

TEMPERATURE = 0.05 # 타우, 높히면 약간의 오차를 더 허용하게 됨
NEG_RATIO = 0.2 # 0.1

LAMBDA = 600.0 # 높히면 semantic, 낮추면 카테고리 분류
# t = 38 # 코사인 유사도 음수 찍는 그 순간임!
# def ALPHA(x): return 1 - 0.3*(2*torch.sigmoid(2.94*/t*steps) - 1) # sigmoid는 1 + math.exp(x) 인거 알지?

In [8]:
# LEARNING_RATE = 1e-3
# BATCH_SIZE = 256
# LAMBDA = 600.0
# TEMPERATURE = 0.05 # 타우, 높히면 약간의 오차를 더 허용하게 됨
# NEG_RATIO = 0.2

In [9]:
# name=f"e5b2-lora-kd-{LAMBDA_VAL}",
# model_name = f"{BATCH_SIZE}({LAMBDA}_{TEMPERATURE}_{NEG_RATIO} Multi-Vector)"
model_name = "GradNorm limit 1000 + s_a"
# lse라 적힌 애들은 neg 0.1로 한거임!!

In [10]:
import wandb

wandb.init(
    project="retrieval-contrastive",
    name = model_name,
    config={
        "lr": LEARNING_RATE,
        "temperature": TEMPERATURE,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
    }
)
# 5d6f712c39c82d6b16acdbb35ab07ced0e27463c

/home/0uk/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/0uk/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using

In [11]:
model_name = "intfloat/e5-small"
# model_name = "intfloat/e5-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

teacher_model = AutoModel.from_pretrained(model_name)
teacher_model.eval()  # 평가 모드 (Dropout 등 비활성화)
for param in teacher_model.parameters():
    param.requires_grad = False # 절대 학습되지 않도록 잠금

# 선생님과 별개의 새로운 인스턴스를 만들어야 함!
student_base_model = AutoModel.from_pretrained(model_name)

"""
Transformer 내부의 특정 Layer(Wq, Wk, Wv 등)에 "LoRA 모듈"을 주입!
E5 Model (pretrained)
    ├── LayerNorm
    ├── 24× Transformer Layers
    │       ├── Attention (Q,K,V,O)
    │       ├── FFN Layers
    │       └── ... (원래 weight 고정)
    └── Pooler → embedding vector   # e.g., CLS embedding

LoRA Injected:
    ├── W_q = W_q + A_q B_q
    ├── W_k = W_k + A_k B_k
    ├── W_v = W_v + A_v B_v
    ├── FFN = FFN + A_ffn B_ffn
    └── (small learnable matrices only)
"""
lora_config = LoraConfig( # Linear(d → d) -→ Linear(d → d) + LoRA(d → d)
    task_type=TaskType.FEATURE_EXTRACTION,  # 임베딩 fine-tuning
    # LoRA가 분류기와 같은 output head에 적용되는 것이 아니라
    # 모델의 Transformer 블록(encoder)에만 적용되도록
    r=16,    # LoRA rank
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)
base_model = get_peft_model(student_base_model, lora_config)
# 원래 weight(W)는 freeze! 훈련 과정에서 업데이트되지 않음.
# LoRA에서 추가된 A, B 행렬만 학습 학습량은 전체의 0.1%~1% 수준으로 줄어듦.
# forward 시에는 LoRA의 low-rank update가 추가됨 ex) outputs = W x + (B(Ax)) * (α / r)

In [12]:
import torch
import torch.nn as nn

"""
# shapes
query = queries           # (B, K, hidden_dim)
key = sequence_output      # (B, L, hidden_dim)
value = sequence_output    # (B, L, hidden_dim)
"""
class E5MultiVectorHead(nn.Module):
    def __init__(self, base_model, num_vectors=3, hidden_dim=384):
        super().__init__()
        self.base_model = base_model # LoRA가 적용된 E5
        self.num_vectors = num_vectors

        # 🎯 핵심: K개의 학습 가능한 쿼리 토큰 (Latent Queries)
        # 예: [Genre_Query, Style_Query, Content_Query]
        self.query_tokens = nn.Parameter(torch.randn(1, num_vectors, hidden_dim))

        # Attention Layer (Transformer Decoder Block과 비슷)
        self.attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)

    def forward(self, input_ids, attention_mask, **kwargs):
        # 1. Base Model (E5) 통과 -> 모든 토큰의 정보 획득
        # outputs shape: (Batch, Seq_Len, Dim)
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state

        # 2. Query 토큰 확장 (Batch Size만큼 복사)
        batch_size = input_ids.shape[0]
        queries = self.query_tokens.repeat(batch_size, 1, 1) # (Batch, K, Dim)

        # 3. Attention Pooling (여기가 마법이 일어나는 곳) 🪄
        # Query: 우리가 궁금한 K개의 주제
        # Key/Value: 책의 전체 내용 (sequence_output)
        # Mask: 패딩 토큰 무시
        key_padding_mask = ~attention_mask.bool()

        # multi_vector shape: (Batch, K, Dim)
        multi_vector, _ = self.attention(
            query=queries,
            key=sequence_output,
            value=sequence_output,
            key_padding_mask=key_padding_mask
        )

        # 4. Normalize (Contrastive Learning을 위해 필수)
        multi_vector = torch.nn.functional.normalize(multi_vector, p=2, dim=2)

        return multi_vector

In [13]:
model = E5MultiVectorHead(base_model, 2, 384)

device = "cuda" if torch.cuda.is_available() else "cpu"
teacher_model.to(device)
model.to(device)

E5MultiVectorHead(
  (base_model): PeftModelForFeatureExtraction(
    (base_model): LoraModel(
      (model): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 384, padding_idx=0)
          (position_embeddings): Embedding(512, 384)
          (token_type_embeddings): Embedding(2, 384)
          (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=384, out_features=384, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(i

In [14]:
# book_path = './data/e5_book_meta.parquet'
book_path = './sample/book_for_e5.parquet'
books = pd.read_parquet(book_path)

In [15]:
def build_text(row): # 입력 텍스트 생성 (타이틀 + 설명 + 저자 등 결합)
    parts = [
        f"Title: {row['title']} |",
        # f"Category: {row['category']} |", # oracle
        f"Description: {row['description']}"
    ]
    return " ".join( # 리스트의 문자열들을 공백으로 연결할건데.....
        [p for p in parts if isinstance(p, str)] # NaN이나 None이 있으면 제외함
    ) # 최종적으로 하나의 문장 형태로 반환한다고 함!! "Title: ... Category: ... Description: ..."

books["text"] = books.apply(build_text, axis=1) # 새 컬럼 text에 대해서.... 문장 만듦

# 100개 미만인 카테고리는 노이즈로 간주하고 삭제
counts = books['category'].value_counts()
valid_categories = counts[counts > 100].index
books = books[books['category'].isin(valid_categories)]

In [16]:
dataset = Dataset.from_pandas(books)

le = LabelEncoder()
le.fit(dataset['category'])   # 전체 데이터로 학습

def encode_label(x):
    return {"label": le.transform([x["category"]])[0]}

dataset = dataset.map(encode_label)

num_classes = len(le.classes_)

Map:   0%|          | 0/81845 [00:00<?, ? examples/s]

collate_fn은 raw text와 label을 텐서로 묶어 모델이 학습할 수 있는 형태로 만들어줌
DataLoader는 이 함수로 미리 전처리한 batch를 모델에 공급하는 역할을 함
```
Dataset row(dict)
     ↓ (DataLoader)
batch = [row1, row2, ...] (list)
     ↓ (collate_fn)
텍스트 리스트 + 라벨 리스트
     ↓ (tokenizer)
input_ids, attention_mask (tensor)
     ↓
(inputs, labels)
     ↓
model(**inputs)
```

In [17]:
# Transformer 모델은 이런 raw 텍스트를 바로 처리 못 하고
# 토크나이저를 거쳐 tensor(batch_input_ids, batch_attention_mask) 형태가 필요함.
def collate_fn(batch): # DataLoader가 batch마다 호출
    # texts = [f"passage: {x['text']}" for x in batch]
    texts = [f"query: {x['text']}" for x in batch]
    labels = torch.tensor([x['label'] for x in batch])  # 라벨을 int 리스트 → torch.tensor 로 변환

    """
    토크나이저:
    텍스트를 token id로 변환 (input_ids), attention_mask 생성,
    batch의 최대 length에 맞춰 패딩, 출력 타입은 PyTorch tensor

    { 'input_ids': tensor([[101,  ... , 102], ...]),
      'attention_mask': tensor([[1,1,1,0,0...], ...) }
    """
    inputs = tokenizer(
      texts, padding=True, truncation=True, max_length=256, return_tensors="pt")

    return inputs, labels

In [18]:
total_len = len(dataset)
train_len = int(total_len * 0.8)
valid_len = total_len - train_len

train_dataset, valid_dataset = random_split(dataset, [train_len, valid_len])

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

지금의 구조는 contrastive loss, genre/content KD, dynamic α, GradNorm, multi-vector head 등이 서로 얽혀 있어 LR 변화에 매우 민감함

- contrastive(genre) 비중이 큰 α=0.8 상태
- genre/content 양방향 KD 압력
- multi-vector head 정렬 전
- GradNorm 비율이 아직 수렴 안 됨
- α decay가 아직 적용 전(장르 비중 과다)

warmup을 너무 짧게(0\~2%) 주면 learning rate가 지나치게 빨리 상승해 초기 단계에서 gradient explosion이나 embedding 붕괴가 일어나고, 반대로 너무 길게(10\~20%) 주면 LR이 천천히 올라가는 동안 contrastive 쪽 표현 학습이 지연되고 embedding collapse 위험이 커져 최종 성능(MRR, top-1)까지 떨어짐

스케쥴러 자체도 바꿨는데, Linear decay는 warmup 이후에 LR이 직선으로 급격히 떨어짐. 이는 학습 후반부 KD alignment와 contrastive alignment의 미세 조정을 막아 학습이 사실상 멈추는 문제가 있음. 반면 Cosine 스케줄러는 warmup 이후 LR을 완만한 곡선 형태로 감소시키기 때문에, 초반에는 안정성을 제공하고, 중반에는 충분한 LR을 유지하며, 후반에도 작은 폭이지만 의미 있는 업데이트를 이어갈 수 있을 것임

In [19]:
total_steps = len(train_loader) * EPOCHS

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
# AdamW 옵티마이저로 LoRA 파라미터만 학습
# LoRA 덕분에 실제 업데이트되는 파라미터는 전체의 1% 정도

# scheduler = get_linear_schedule_with_warmup(
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.05), # WARMUP_RATIO), 원래 linear 일때 썻던 값은데 0.1이었거든? 0.05로 줄이래
    num_training_steps=total_steps,
)

In [20]:
import torch
import torch.nn.functional as F

def compute_retrieval_accuracy(model, dataloader, device, tokenizer, k=10):
    """
    Multi-Vector 모델용 검증 함수
    * Vector 0 (Genre Vector)만 사용하여 카테고리 검색 성능을 측정합니다.
    """
    embeddings_list = []
    labels_list = []

    model.eval() # 평가 모드 필수!

    with torch.no_grad():
        for batch_inputs, labels in dataloader:
            # 1. 입력 데이터 준비
            # input_ids = batch_inputs.to(device)
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            labels = labels.to(device)

            # 2. 모델 Forward (Multi-Vector 출력)
            # output shape: (Batch, K, 768)
            multi_vectors = model(**batch_inputs)

            # 3. ★핵심★: Vector 0 (장르 벡터)만 추출
            # 우리가 검증하려는 건 "카테고리(장르)를 잘 맞추는가?" 이므로 0번만 씀
            genre_embeddings = multi_vectors[:, 0, :]
            genre_embeddings = F.normalize(genre_embeddings, p=2, dim=1)

            embeddings_list.append(genre_embeddings)
            labels_list.append(labels)

    # 전체 데이터 합치기
    all_embeddings = torch.cat(embeddings_list, dim=0)
    all_labels = torch.cat(labels_list, dim=0)
    similarity = torch.matmul(all_embeddings, all_embeddings.T) # (N, N) 행렬 계산
    similarity.fill_diagonal_(-1e9) # 자기 자신 제외 (-무한대 마스킹)

    _, topk_idx = similarity.topk(k, dim=1)  # top-k neighbor 인덱스
    nn_labels_topk = all_labels[topk_idx] # 이웃들의 라벨 가져오기 (N, k)

    # 정답 여부 행렬 (True/False)
    # 내 라벨(all_labels)과 이웃 라벨(nn_labels_topk)이 같은지 비교
    # unsqueeze(1)로 (N, 1) vs (N, k) 브로드캐스팅 비교
    hits = (nn_labels_topk == all_labels.unsqueeze(1)) # (N, k) Bool Tensor

    # -----------------------------
    # top-1: 가장 가까운 neighbor 정답 여부
    # top-k: k개 안에 정답이 하나라도 있으면
    # precision@k: k개 neighbor 중 정답 비율 평균
    # MRR: 정답이 rank 몇 번째인지에 따른 평균 역수 → rank 1이면 1.0, rank 2이면 0.5
    # -----------------------------
    # Top-1 accuracy
    top1_labels = nn_labels_topk[:, 0]
    top1_acc = (top1_labels == all_labels).float().mean().item()

    # Top-k accuracy (at least 1 match)
    topk_match = (nn_labels_topk == all_labels.unsqueeze(1)).any(dim=1).float()
    topk_acc = topk_match.mean().item()

    # Precision@k
    precision_at_k = (nn_labels_topk == all_labels.unsqueeze(1)).float().mean(dim=1).mean().item()

    # MRR (Mean Reciprocal Rank)
    ranks = (nn_labels_topk == all_labels.unsqueeze(1)).float() # 정답 label 위치 찾기
    reciprocal_rank = []
    for i in range(ranks.size(0)):
        pos_positions = torch.nonzero(ranks[i]).flatten()
        if len(pos_positions) == 0: # positive 없으면 reciprocal rank = 0
            reciprocal_rank.append(0.0)
        else:
            reciprocal_rank.append(1.0 / (pos_positions[0].item() + 1))
    mrr = sum(reciprocal_rank) / len(reciprocal_rank)

    metrics = {
        "top1_acc": top1_acc,
        "topk_acc": topk_acc,
        "precision@k": precision_at_k,
        "mrr": mrr
    }

    return metrics

In [21]:
# 일단 한번 해보고 개선되면 memory bank 식으로 바꿔보자
# 근데 배치가 크면 굳이 memory bank로 안바꿔도 된다는디?
def hard_negative(embeddings, labels, neg_ratio=0.1): # k=3):
    batch_size = embeddings.size(0)
    k = max(3, int(batch_size * neg_ratio)) # 배치 사이즈에 따라 K값 결정

    # 1. 유사도 행렬 계산
    similarity = torch.matmul(embeddings, embeddings.T)

    # hard_neg_sims = []
    # for i in range(batch_size): # for문 대신 행렬 연산(Vectorization)으로 바꾸면 빨라진대
    #     mask = labels != labels[i]           # 다른 라벨만
    #     sim_row = similarity[i].clone()
    #     sim_row[~mask] = -1e9                # 같은 라벨 제외
    #     topk_sim, _ = sim_row.topk(k)        # 가장 가까운 k개
    #     hard_neg_sims.append(topk_sim)
    # hard_neg_sims = torch.stack(hard_neg_sims, dim=0)  # (batch_size, k)

    # 2. Positive Mask 생성 (같은 라벨인 곳 True)
    # labels: (N) -> (N, 1) == (1, N) 브로드캐스팅
    pos_mask = labels.unsqueeze(1) == labels.unsqueeze(0)

    # 3. 같은 라벨(Positive + 자기자신)은 유사도를 -무한대로 보내서 topk에서 제외
    # clone()을 안 쓰면 원본 similarity가 망가지므로 주의 (필요시)
    sim_for_neg = similarity.clone()
    sim_for_neg.masked_fill_(pos_mask, -1e9)  # 같은 라벨 제외

    # 4. 각 행(Query)마다 가장 높은 유사도를 가진 k개(Hard Negatives) 추출
    hard_neg_sims, _ = sim_for_neg.topk(k, dim=1) # (N, k)

    return hard_neg_sims

In [22]:
def noisy_contrastive_loss(embeddings, labels):
    # https://www.blossominkyung.com/deeplearning/contrastive-learning
    # 같은 카테고리는 가까이, 다른 카테고리는 멀리
    similarity = torch.matmul(embeddings, embeddings.T)

    # Mask 생성
    labels_eq = labels.unsqueeze(1) == labels.unsqueeze(0)
    identity_mask = torch.eye(len(labels), device=labels.device).bool() # 자기 자신 제거 mask
    labels_eq = labels_eq & (~identity_mask)

    pos_mask = labels_eq.float()   # (N, N)

    """
    pos_sim: anchor와 positive 샘플들의 유사도 벡터
    → 한 anchor가 여러 개의 positive를 가질 수 있음

    neg_sim: anchor와 negative 샘플들의 유사도 벡터
    → 당연히 여러 negative가 존재
    """
    # 정답이 아닌 곳(0)에는 아주 작은 값(-1e9)을 넣어서 exp 계산 시 0이 되도록 해야 함
    # pos_sim = similarity.masked_fill(~pos_mask.bool(), -1e9)
    pos_sim = similarity * pos_mask # 이거 넣으면... 정답 아닌 곳 => 0 => e^0=1 => 1이 쌓인다는데....
    # 난 그거 더 성능이 좋게 나오던데????? 수학적으론 틀려도.....
    # 노이즈가 섞여있어서 한 점으로 덜 뭉쳐서 뭐 retrieval이 더 좋게 나온다던데..?
    """
    exp(0)=1이 일정한 baseline noise 역할
    positive가 너무 한 점으로 collapse 되는 것을 방지
    embedding space가 overly tight cluster로 수렴하지 않음
    retrieval에서는 적절한 분산이 성능을 올리는 경우가 많음
    즉, “정석적인 contrastive”에서는 틀렸지만
    retrieval task에서는 종종 regularization 역할이 되어 더 잘되는 경우가 많음.
    """
    neg_sim = hard_negative(embeddings, labels, neg_ratio=NEG_RATIO) # (N, k) → anchor i의 negative k개

    # loss 확대: 정답(0.8/0.05=16), 오답(0.7/0.05=14) => exp(16) ≈ 8,886,110 vs exp(14) ≈ 1,202,604 7배 이상 차이남
    # => 0.1 차이도 크게 만들어 모델이 pos를 더더더 1에 가깝도록 맞춤
    pos_sim = pos_sim / TEMPERATURE # 아니 근데 생각해보니까 noise는 /temp 전에 들어가잖아?
    neg_sim = neg_sim / TEMPERATURE

    # 모든 positive를 합산해서 ratio 계산 → 카테고리별 embedding을 한 점에 모으는 경향
    loss = -torch.log(
        torch.exp(pos_sim).sum(dim=1) /
        (torch.exp(pos_sim).sum(dim=1) + torch.exp(neg_sim).sum(dim=1))
    ).mean()
    # sum해주는 이유는.. 한 anchor(기준 샘플)당 여러 개의 pos/neg 쌍이 존재할 수 있기 때문
    # 그림 mean은 왜 하는거지 ㅋㅋ..

    return loss

In [23]:
import torch
import torch.nn.functional as F

def compute_multivector_diagnostics(model, l_cont, l_kd_genre, l_kd_content, lambda_genre=50.0, lambda_content=500.0):
    """
    Multi-Vector 학습의 '힘의 균형'을 분석하는 도구
    """
    # 1. 학습 대상 파라미터만 추출 (LoRA + Head)
    # requires_grad=True인 것만 가져와야 None 에러가 안 남
    params = [p for p in model.parameters() if p.requires_grad]

    # 2. 각각의 Loss에 대한 Gradient 추출 (Retain Graph 필수!)
    # 메모리를 좀 먹으니, 진단은 100 step마다 한 번씩만 하길 권장
    grads_cont = torch.autograd.grad(l_cont, params, retain_graph=True, allow_unused=True)
    grads_kd_g = torch.autograd.grad(l_kd_genre, params, retain_graph=True, allow_unused=True)
    grads_kd_c = torch.autograd.grad(l_kd_content, params, retain_graph=True, allow_unused=True)

    # 3. Flatten Helper
    def get_flat_grad(grads):
        vec = []
        for g in grads:
            if g is not None:
                vec.append(g.view(-1))
        return torch.cat(vec) if vec else None

    v_cont = get_flat_grad(grads_cont)
    v_kd_g = get_flat_grad(grads_kd_g)
    v_kd_c = get_flat_grad(grads_kd_c)

    if v_cont is None or v_kd_g is None or v_kd_c is None:
        return {}

    # -------------------------------------------------------
    # 4. 분석: 힘의 크기 (Magnitude) 비교
    # -------------------------------------------------------

    # A. 장르 벡터의 추진력 vs 저항력
    # Norm 비교: ||G_cont|| vs ||50 * G_kd_genre||
    norm_cont = v_cont.norm().item()
    norm_drag = v_kd_g.norm().item() * lambda_genre

    # B. 내용 벡터의 보존력
    # Norm 비교: ||500 * G_kd_content||
    norm_anchor = v_kd_c.norm().item() * lambda_content

    # -------------------------------------------------------
    # 5. 분석: 힘의 방향 (Direction) 비교
    # -------------------------------------------------------

    # A. 장르 내부 갈등 (Classification vs Regularization)
    # 음수가 나오면 서로 반대 방향 (정상)
    cos_friction = F.cosine_similarity(v_cont.unsqueeze(0), v_kd_g.unsqueeze(0)).item()

    # B. 메인 충돌 (Classification vs Content Preservation)
    # 이게 가장 중요함. 서로 반대 방향일 때 힘의 크기가 비슷해야 붕괴를 막음.
    cos_conflict = F.cosine_similarity(v_cont.unsqueeze(0), v_kd_c.unsqueeze(0)).item()

    return {
        "Force(Cluster)": norm_cont,       # 클러스터링 하려는 힘
        "Force(Drag)": norm_drag,          # 장르 이탈 막는 약한 힘 (50)
        "Force(Anchor)": norm_anchor,      # 내용 지키는 강한 힘 (500)
        "Cos(Friction)": cos_friction,     # 장르 내부 각도
        "Cos(Conflict)": cos_conflict      # 장르 vs 내용 각도
    }

grad_logs = {
    "force_cluster": [],
    "force_drag": [],
    "force_anchor": [],
    "cos_friction": [],
    "cos_conflict": [],
    "step": []
}
lora_params = [p for p in model.parameters() if p.requires_grad]

In [24]:
# --------------------------------------------------------------------------
# 1. GradNorm 계산 함수 정의
# --------------------------------------------------------------------------
def calc_grad_norm(loss, model_layer):
    """
    특정 Loss가 특정 레이어(model_layer)의 파라미터에 가하는
    Gradient의 총량(Norm)을 계산합니다.
    """
    # 1. 해당 레이어의 파라미터만 가져옴 (requires_grad=True인 것만)
    params = [p for p in model_layer.parameters() if p.requires_grad]

    if not params:
        return 0.0

    # 2. Gradient 계산 (create_graph=False, retain_graph=True 필수!)
    # retain_graph=True: 뒤에 진짜 backward()를 또 해야 하므로 그래프를 날리면 안 됨
    grads = torch.autograd.grad(
        loss,
        params,
        retain_graph=True,
        allow_unused=True
    )

    # 3. Norm(크기) 합산 (L2 Norm)
    total_norm = 0.0
    for g in grads:
        if g is not None:
            total_norm += g.pow(2).sum().item()

    return total_norm ** 0.5

In [25]:
ㅋㅋ

NameError: name 'ᄏᄏ' is not defined

In [ ]:
metrics = compute_retrieval_accuracy(model, valid_loader, device, tokenizer)
wandb.log({
    "epoch": 0,
    "valid/top1_acc": metrics["top1_acc"],
    "valid/top10_acc": metrics["topk_acc"],
    "valid/precision@10": metrics["precision@k"],
    "valid/mrr": metrics["mrr"],
})
print(f"[Before Training...]")
print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR             : {metrics['mrr']:.4f}")


t = 200  # 충돌 시작 지점
base_alpha = 0.8   # 초반: 장르(Genre) 정보를 확실히 잡음
target_alpha = 0.2 # 후반: 충돌 회피를 위해 장르 비중을 낮춤 (본문 집중)
steps_per_epoch = len(train_loader)
running_ratio = 1.0
beta = 0.95  # 관성 계수 (클수록 변화가 부드러움)

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    genre_loss_cont = 0                      # 디버깅
    genre_loss_kd = 0                        # 디버깅
    content_loss_kd = 0                      # 디버깅
    epoch_loss_sum = 0.0                     # 디버깅
    epoch_norm_main_sum = 0.0                # 디버깅
    epoch_norm_sub_sum = 0.0                 # 디버깅
    epoch_ratio_sum = 0.0                    # 디버깅

    for step, (batch_inputs, labels) in enumerate(tqdm(train_loader, desc = f"Epoch: {epoch+1}")):
        # global_step = epoch * steps_per_epoch + step

        # # --- 1. Alpha Scheduling (Decay 전략 적용) ---
        # # 충돌 시점(t) 이후부터 장르 비중(alpha)을 줄여서 충돌을 피합니다.
        # if global_step < t:
        #     alpha = base_alpha
        # else:
        #     # t 이후부터 Alpha Decay (장르 비중 감소)
        #     # global_step이 t보다 커질수록 x가 증가 -> sigmoid 증가 -> alpha 감소
        #     # x 계산: 변화 속도 조절 (숫자 5는 기울기, 필요시 조절)
        #     x = 5 * (global_step - t) / t
        #     sigmoid_x = 1 / (1 + math.exp(-x))

        #     # sigmoid_x는 0.5부터 시작해서 1.0으로 감
        #     # 이를 0.0 ~ 1.0 비율로 정규화하여 사용
        #     decay_ratio = (sigmoid_x - 0.5) * 2
        #     if decay_ratio > 1.0: decay_ratio = 1.0

        #     # base(0.8)에서 target(0.2)으로 서서히 이동
        #     alpha = base_alpha - (base_alpha - target_alpha) * decay_ratio
        #     # # (혹은 간단하게 step > t 일 때 바로 target_alpha로 고정해도 효과는 봅니다)
        #     # if alpha < target_alpha: alpha = target_alpha
        alpha = 0.8

        # 학습 데이터 gpu에 올리기
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        labels = labels.to(device)

        # 2. Teacher (Original E5) Forward
        with torch.no_grad(): # 원본 모델에게 물어봅니다. "너라면 이거 어디에 배치할래?"
            teacher_outputs = teacher_model(**batch_inputs)

            # teacher_embeddings = teacher_outputs.last_hidden_state.mean(dim=1)
            hidden = teacher_outputs.last_hidden_state       # (B, L, D)
            mask = batch_inputs['attention_mask'].unsqueeze(-1)  # (B, L, 1)
            teacher_embeddings = (hidden * mask).sum(dim=1) / mask.sum(dim=1)

            teacher_embeddings = F.normalize(teacher_embeddings, p=2, dim=1)

        # 3. Student LoRA Forward
        student_vectors = model(**batch_inputs) # B, 3, D
        """
        model(**batch_inputs) 은 자동으로 다음과 같이 해석됨:
        model(
            input_ids=batch_inputs["input_ids"],
            attention_mask=batch_inputs["attention_mask"]
        )
        """

        # 4. 벡터 쪼개기
        genre_vector = student_vectors[:, 0, :] # Vector 0: 장르
        content_vectors = student_vectors[:, 1:, :] # Vector 1: 내용

        # --- Loss Calculation Start ---
        """
        1. loss_contrastive (카테고리 로스): "같은 카테고리니까 뭉쳐! 다른 카테고리는 저리 가!"
            → 힘의 방향: 공간 왜곡 (Distortion)

        2. loss_distill (보존 로스): "야, 그래도 원래 의미(Semantic)가 '우주'랑 'SF'였잖아. 그 근처에 있어."
            → 힘의 방향: 원래 자리 유지 (Preservation)

        결과적으로 모델은 원래 의미 공간을 최대한 유지하면서, 카테고리 분류가 가능할 정도로만 살짝 이동하게 됨
        """
        # 1. supervised contrastive sum over positives
        loss_cont = noisy_contrastive_loss(genre_vector, labels)
        # genre_vector는 이미 E5MultiVectorHead에서 정규화 했으니 ㄱㅊ
        # student_vector 자체가 이미 길이가 1인 벡터임

        # 1-2. ★추가됨★: 장르 벡터 폭주 방지용 KD (Safety Net)
        # 정규화(Normalize) 후 MSE 계산? 왜냐만 길이가 1인 벡터를 평균내서 l2-norm 구하면 1보다 작아지잖아
        # (1,0), (0,1) -> (0.5, 0), (0, 0.5) -> l2-norm(길이) = 0.707
        genre_norm = F.normalize(genre_vector, p=2, dim=1)
        loss_kd_genre = F.mse_loss(genre_norm, teacher_embeddings)

        # 2. distillation loss (MSE)
        # 카테고리 맞추는 것도 좋은데, 원래 위치랑 너무 달라지진 마
        student_content_mean = content_vectors.mean(dim=1)
        student_content_mean = F.normalize(student_content_mean, p=2, dim=1)
        loss_kd_content = F.mse_loss(student_content_mean, teacher_embeddings)

        # KD Loss 결합 (Alpha 적용)
        loss_kd_combined = alpha * loss_kd_genre + (1 - alpha) * loss_kd_content

        # --- 2. Dynamic Weighting (GradNorm 측정 방식) ---
        # 1. Main Task가 어텐션 레이어를 당기는 힘 측정
        norm_main = calc_grad_norm(loss_cont, model.attention)

        # 2. Sub Task가 어텐션 레이어를 당기는 힘 측정
        norm_sub = calc_grad_norm(loss_kd_combined, model.attention)

        # 3. 비율 계산 (Sub의 힘이 0이면 나눗셈 에러 나니까 1e-8 추가)
        # 목표: norm_sub * ratio가 norm_main과 비슷해지도록 함
        # 즉, ratio = norm_main / norm_sub
        # current_ratio = norm_main / (norm_sub + 1e-8)

        # 3 수정
        # 원래 GradNorm: ratio = Main / Sub  ->  결과: Main 힘 == Sub 힘 (1:1)
        # 수정 GradNorm: ratio = (Main / Sub) * 0.2  -> 결과: Sub 힘이 Main의 20%가 됨 (5:1)



        global_step = epoch * steps_per_epoch + step

        # --- 1. Alpha Scheduling (Decay 전략 적용) ---
        # 충돌 시점(t) 이후부터 장르 비중(alpha)을 줄여서 충돌을 피합니다.
        if global_step < t:
            decay_ratio = 1.0
        else:
            # t 이후부터 Alpha Decay (장르 비중 감소)
            # global_step이 t보다 커질수록 x가 증가 -> sigmoid 증가 -> alpha 감소
            # x 계산: 변화 속도 조절 (숫자 5는 기울기, 필요시 조절)
            x = 5 * (global_step - t) / t
            sigmoid_x = 1 / (1 + math.exp(-x))

            # sigmoid_x는 0.5부터 시작해서 1.0으로 감
            decay_ratio = (-1) * sigmoid_x + 1.5 # 얜 그럼 1.0 -> 0.5

        current_ratio = norm_main / (norm_sub + 1e-8) * decay_ratio

        # 4. 이동 평균 업데이트 (학습 안정화)
        running_ratio = beta * running_ratio + (1 - beta) * current_ratio

        # 최종 Loss: 이제 두 항의 스케일(Magnitude)이 1:1로 맞춰집니다.
        total_loss = loss_cont + running_ratio * loss_kd_combined




        """++++++++++++++++++++++++++++++++++++++++디버깅++++++++++++++++++++++++++++++++++++++++"""
        # real_lambda_genre = LAMBDA * alpha
        # real_lambda_content = LAMBDA * (1 - alpha)
        real_lambda_genre = running_ratio * alpha
        real_lambda_content = running_ratio * (1 - alpha)
        if step % 100 == 0:
            # 1. 진단 함수 호출 (딕셔너리 리턴)
            # 주의: .item()이 아니라 Tensor 상태의 loss 변수를 넣어야 합니다!
            diag_results = compute_multivector_diagnostics(
                model,
                l_cont=loss_cont,
                l_kd_genre=loss_kd_genre,
                l_kd_content=loss_kd_content,
                lambda_genre=real_lambda_genre,
                lambda_content=real_lambda_content
            )
            # A. 힘의 크기 (Norm)
            grad_logs["force_cluster"].append(diag_results["Force(Cluster)"])
            grad_logs["force_drag"].append(diag_results["Force(Drag)"])
            grad_logs["force_anchor"].append(diag_results["Force(Anchor)"])

            # B. 힘의 방향 (Cosine)
            grad_logs["cos_friction"].append(diag_results["Cos(Friction)"])
            grad_logs["cos_conflict"].append(diag_results["Cos(Conflict)"])

            grad_logs["step"].append(step + epoch * len(train_loader))
        """++++++++++++++++++++++++++++++++++++++++디버깅++++++++++++++++++++++++++++++++++++++++"""


        # 역전파 과정
        # total_loss.backward()  # 기울기(Gradient) 계산 완료
        # optimizer.zero_grad()  # 기울기를 0으로 초기화 (삭제) 😱
        # optimizer.step()       # 0이 된 기울기로 가중치 업데이트 (변화 없음)
        optimizer.zero_grad()  # 1. 이전 기울기 청소 이거 맨 위로 옮김
        total_loss.backward()  # 2. 현재 기울기 계산
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)    # 테스트임!!!!
        optimizer.step()       # 3. 업데이트

        scheduler.step() # 스케줄러가 step 단위라면 여기, epoch 단위면 loop 밖으로

        total_train_loss += total_loss.item()
        genre_loss_cont += loss_cont.item()                   # 디버깅
        genre_loss_kd += loss_kd_genre.item()                 # 디버깅
        content_loss_kd += loss_kd_content.item()             # 디버깅
        epoch_loss_sum += total_loss.item()                   # 디버깅
        epoch_norm_main_sum += norm_main                      # 디버깅
        epoch_norm_sub_sum += norm_sub                        # 디버깅
        epoch_ratio_sum += running_ratio                      # 디버깅

    train_loss = total_train_loss / len(train_loader)
    train_genre_loss_cont = genre_loss_cont / len(train_loader)     # 디버깅
    train_genre_loss_kd = genre_loss_kd / len(train_loader)         # 디버깅
    train_content_loss_kd = content_loss_kd / len(train_loader)     # 디버깅
    avg_loss = epoch_loss_sum / len(train_loader)                   # 디버깅
    avg_norm_main = epoch_norm_main_sum / len(train_loader)         # 디버깅
    avg_norm_sub = epoch_norm_sub_sum / len(train_loader)           # 디버깅
    avg_ratio = epoch_ratio_sum / len(train_loader)                 # 디버깅

    model.eval()
    metrics = compute_retrieval_accuracy(model, valid_loader, device, tokenizer)
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "genre_loss_cont": train_genre_loss_cont,
        "genre_loss_kd": train_genre_loss_kd,
        "content_loss_kd": train_content_loss_kd,
        "train/grad_ratio_avg": avg_ratio,      # 이번 에포크 평균 비율
        "train/norm_main_avg": avg_norm_main,   # 메인 태스크 그래디언트 평균 크기
        "train/norm_sub_avg": avg_norm_sub,     # 서브 태스크 그래디언트 평균 크기
        "valid/top1_acc": metrics["top1_acc"],
        "valid/precision@10": metrics["precision@k"],
        "valid/mrr": metrics["mrr"],
        "lr": scheduler.get_last_lr()[0],
    })
    print(f"[Epoch {epoch + 1}] Train Loss: {train_loss:.4f}")
    print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
    print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR              : {metrics['mrr']:.4f}")

In [ ]:
import os
save_path = f"./old_multi_vec_{LAMBDA}_20ep.pth"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(model.state_dict(), save_path)

In [ ]:
import matplotlib.pyplot as plt

"""
grad_logs 딕셔너리에 저장된 Multi-Vector 학습 진단 데이터를 시각화합니다.
Keys expected: 'step', 'cos_friction', 'cos_conflict',
               'force_cluster', 'force_drag', 'force_anchor'
"""
steps = grad_logs["step"]

# 데이터를 CPU로 옮기는 헬퍼 함수
def to_cpu(data_list):
    return [x.item() if torch.is_tensor(x) else x for x in data_list]

# 그래프 스타일 설정 (가독성 향상)
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12), sharex=True)

# ---------------------------------------------------------
# 1. Cosine Similarity (방향성) 그래프
# ---------------------------------------------------------
# Cos(Friction): 장르 벡터 내부의 방향 (Cont vs KD_Genre)
ax1.plot(steps, grad_logs["cos_friction"],
         label="Cos(Friction): Genre Internal", color='green', alpha=0.7, linestyle='--')

# Cos(Conflict): 메인 충돌 (Genre vs Content) -> 이게 가장 중요!
ax1.plot(steps, grad_logs["cos_conflict"],
         label="Cos(Conflict): Genre vs Content", color='red', linewidth=2)

# 기준선 (0.0)
ax1.axhline(0, color='black', linewidth=1, linestyle='-')

ax1.set_title("1. Gradient Direction Analysis (Cosine Similarity)", fontsize=14, fontweight='bold')
ax1.set_ylabel("Cosine Similarity (-1 ~ 1)", fontsize=12)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, linestyle='--', alpha=0.6)

# ---------------------------------------------------------
# 2. Gradient Norms (힘의 크기) 그래프
# ---------------------------------------------------------
# Force(Cluster): 장르를 나누려는 힘 (Cont)
ax2.plot(steps, grad_logs["force_cluster"],
         label="Force(Cluster): ||G_cont||", color='blue', linewidth=2)
# 2. Anchor (Orange) - 내용을 지키려는 힘 (500 * KD_Content) -> 빨간색이 파란색을 감싸야 함
ax2.plot(steps, to_cpu(grad_logs["force_anchor"]),
         label="Force(Anchor): ||λ*α*G_kd_c||", color='orange', linewidth=2, linestyle='-')

# 3. Drag (Green) 장르 이탈을 막는 약한 힘 (50 * KD_Genre) -> 바닥에 깔려야 함
ax2.plot(steps, to_cpu(grad_logs["force_drag"]),
         label="Force(Drag): ||λ*(1-α)*G_kd_g||", color='green', alpha=0.6, linestyle=':')

ax2.set_title("2. Gradient Magnitude Analysis (Forces)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Training Step", fontsize=12)
ax2.set_ylabel("Gradient Norm", fontsize=12)
ax2.legend(loc='upper left', fontsize=10)
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
import os
save_path = f"./new_multi_vec_{LAMBDA}_20ep.pth"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(model.state_dict(), save_path)

### 아랜 원래 학습코드

In [ ]:
다이나믹한 alpha(sigmoid)

In [ ]:
metrics = compute_retrieval_accuracy(model, valid_loader, device, tokenizer)
wandb.log({
    "epoch": 0,
    "valid/top1_acc": metrics["top1_acc"],
    "valid/top10_acc": metrics["topk_acc"],
    "valid/precision@10": metrics["precision@k"],
    "valid/mrr": metrics["mrr"],
})
print(f"[Before Training...]")
print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR             : {metrics['mrr']:.4f}")

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    genre_loss_cont = 0                      # 디버깅
    genre_loss_kd = 0                        # 디버깅
    content_loss_kd = 0                      # 디버깅

    # ratio = math.sin(math.pi * epoch / (2 * EPOCHS))
    t = 200 # 코사인 유사도 음수 찍는 그 순간임!
    base_alpha = 0.3
    target_alpha = 0.7
    for step, (batch_inputs, labels) in enumerate(tqdm(train_loader, desc = f"Epoch: {epoch+1}")):
        x = 3 * step / t
        sigmoid_x = 1 / (1 + math.exp(-x))
        growth_ratio = 2 * sigmoid_x - 1  # 0 ~ 1
        alpha = base_alpha + (target_alpha - base_alpha) * growth_ratio

        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        labels = labels.to(device)

        # 1. Optimizer 초기화 (가장 먼저!)
        optimizer.zero_grad()

        # 2. Teacher (Original E5) Forward
        with torch.no_grad(): # 원본 모델에게 물어봅니다. "너라면 이거 어디에 배치할래?"
            teacher_outputs = teacher_model(**batch_inputs)

            # teacher_embeddings = teacher_outputs.last_hidden_state.mean(dim=1)
            hidden = teacher_outputs.last_hidden_state       # (B, L, D)
            mask = batch_inputs['attention_mask'].unsqueeze(-1)  # (B, L, 1)
            teacher_embeddings = (hidden * mask).sum(dim=1) / mask.sum(dim=1)

            teacher_embeddings = F.normalize(teacher_embeddings, p=2, dim=1)

        # 3. Student LoRA Forward
        student_vectors = model(**batch_inputs) # B, 3, D
        """
        model(**batch_inputs) 은 자동으로 다음과 같이 해석됨:
        model(
            input_ids=batch_inputs["input_ids"],
            attention_mask=batch_inputs["attention_mask"]
        )
        """

        # 4. 벡터 쪼개기
        genre_vector = student_vectors[:, 0, :] # Vector 0: 장르
        content_vectors = student_vectors[:, 1:, :] # Vector 1: 내용

        # --- Loss Calculation Start ---
        """
        1. loss_contrastive (카테고리 로스): "같은 카테고리니까 뭉쳐! 다른 카테고리는 저리 가!"
            → 힘의 방향: 공간 왜곡 (Distortion)

        2. loss_distill (보존 로스): "야, 그래도 원래 의미(Semantic)가 '우주'랑 'SF'였잖아. 그 근처에 있어."
            → 힘의 방향: 원래 자리 유지 (Preservation)

        결과적으로 모델은 원래 의미 공간을 최대한 유지하면서, 카테고리 분류가 가능할 정도로만 살짝 이동하게 됨
        """
        # 1. supervised contrastive sum over positives
        loss_cont = noisy_contrastive_loss(genre_vector, labels)
        # genre_vector는 이미 E5MultiVectorHead에서 정규화 했으니 ㄱㅊ
        # student_vector 자체가 이미 길이가 1인 벡터임

        # 1-2. ★추가됨★: 장르 벡터 폭주 방지용 KD (Safety Net)
        # 정규화(Normalize) 후 MSE 계산? 왜냐만 길이가 1인 벡터를 평균내서 l2-norm 구하면 1보다 작아지잖아
        # (1,0), (0,1) -> (0.5, 0), (0, 0.5) -> l2-norm(길이) = 0.707
        genre_norm = F.normalize(genre_vector, p=2, dim=1)
        loss_kd_genre = F.mse_loss(genre_norm, teacher_embeddings)

        # 2. distillation loss (MSE)
        # 카테고리 맞추는 것도 좋은데, 원래 위치랑 너무 달라지진 마
        student_content_mean = content_vectors.mean(dim=1)
        student_content_mean = F.normalize(student_content_mean, p=2, dim=1)

        loss_kd_content = F.mse_loss(student_content_mean, teacher_embeddings)

        # 3. Loss 합산
        # - loss_cont: 그대로 둠 (1.0)
        # - loss_kd_genre: 50.0 (약한 제약. 카테고리 이동을 허용하되 폭발만 막음)
        # - loss_kd_content: 500.0 (강한 제약. 내용 보존 필수)
        total_loss = loss_cont + LAMBDA*(alpha*loss_kd_genre + (1-alpha)*loss_kd_content)
        # 초반엔 loss_kd_genre에 더 큰 람다가 곱해져야함




        """++++++++++++++++++++++++++++++++++++++++디버깅++++++++++++++++++++++++++++++++++++++++"""
        if step % 100 == 0:
            # 1. 진단 함수 호출 (딕셔너리 리턴)
            # 주의: .item()이 아니라 Tensor 상태의 loss 변수를 넣어야 합니다!
            diag_results = compute_multivector_diagnostics(
                model,
                l_cont=loss_cont,
                l_kd_genre=loss_kd_genre,
                l_kd_content=loss_kd_content,
                lambda_genre=50.0,       # 자네가 설정한 값
                lambda_content=LAMBDA # 500.0
            )
            # A. 힘의 크기 (Norm)
            grad_logs["force_cluster"].append(diag_results["Force(Cluster)"])
            grad_logs["force_drag"].append(diag_results["Force(Drag)"])
            grad_logs["force_anchor"].append(diag_results["Force(Anchor)"])

            # B. 힘의 방향 (Cosine)
            grad_logs["cos_friction"].append(diag_results["Cos(Friction)"])
            grad_logs["cos_conflict"].append(diag_results["Cos(Conflict)"])

            grad_logs["step"].append(step + epoch * len(train_loader))
        """++++++++++++++++++++++++++++++++++++++++디버깅++++++++++++++++++++++++++++++++++++++++"""



        # total_loss.backward()  # 기울기(Gradient) 계산 완료
        # optimizer.zero_grad()  # 기울기를 0으로 초기화 (삭제) 😱
        # optimizer.step()       # 0이 된 기울기로 가중치 업데이트 (변화 없음)
        optimizer.zero_grad()  # 1. 이전 기울기 청소
        total_loss.backward()  # 2. 현재 기울기 계산
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)    # 테스트임!!!!
        optimizer.step()       # 3. 업데이트

        scheduler.step() # 스케줄러가 step 단위라면 여기, epoch 단위면 loop 밖으로

        total_train_loss += total_loss.item()
        genre_loss_cont += loss_cont.item()                   # 디버깅
        genre_loss_kd += loss_kd_genre.item()                 # 디버깅
        content_loss_kd += loss_kd_content.item()             # 디버깅

    train_loss = total_train_loss / len(train_loader)
    train_genre_loss_cont = genre_loss_cont / len(train_loader)     # 디버깅
    train_genre_loss_kd = genre_loss_kd / len(train_loader)         # 디버깅
    train_content_loss_kd = content_loss_kd / len(train_loader)     # 디버깅

    model.eval()
    metrics = compute_retrieval_accuracy(model, valid_loader, device, tokenizer)
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "genre_loss_cont": train_genre_loss_cont,
        "genre_loss_kd": train_genre_loss_kd,
        "content_loss_kd": train_content_loss_kd,
        "valid/top1_acc": metrics["top1_acc"],
        "valid/precision@10": metrics["precision@k"],
        "valid/mrr": metrics["mrr"],
        "lr": scheduler.get_last_lr()[0],
    })
    print(f"[Epoch {epoch + 1}] Train Loss: {train_loss:.4f}")
    print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
    print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR              : {metrics['mrr']:.4f}")

In [ ]:
alpha 0.2, Gradient에 sigmoid 붙힘

In [26]:
# metrics = compute_retrieval_accuracy(model, valid_loader, device, tokenizer)
# wandb.log({
#     "epoch": 0,
#     "valid/top1_acc": metrics["top1_acc"],
#     "valid/top10_acc": metrics["topk_acc"],
#     "valid/precision@10": metrics["precision@k"],
#     "valid/mrr": metrics["mrr"],
# })
# print(f"[Before Training...]")
# print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
# print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR             : {metrics['mrr']:.4f}")


t = 200  # 충돌 시작 지점
base_alpha = 0.8   # 초반: 장르(Genre) 정보를 확실히 잡음
target_alpha = 0.2 # 후반: 충돌 회피를 위해 장르 비중을 낮춤 (본문 집중)
steps_per_epoch = len(train_loader)
running_ratio = 1.0
beta = 0.95  # 관성 계수 (클수록 변화가 부드러움)

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    genre_loss_cont = 0                      # 디버깅
    genre_loss_kd = 0                        # 디버깅
    content_loss_kd = 0                      # 디버깅
    epoch_loss_sum = 0.0                     # 디버깅
    epoch_norm_main_sum = 0.0                # 디버깅
    epoch_norm_sub_sum = 0.0                 # 디버깅
    epoch_ratio_sum = 0.0                    # 디버깅

    for step, (batch_inputs, labels) in enumerate(tqdm(train_loader, desc = f"Epoch: {epoch+1}")):
        global_step = epoch * steps_per_epoch + step

        # --- 1. Alpha Scheduling (Decay 전략 적용) ---
        # 충돌 시점(t) 이후부터 장르 비중(alpha)을 줄여서 충돌을 피합니다.
        if global_step < t:
            alpha = base_alpha
        else:
            # t 이후부터 Alpha Decay (장르 비중 감소)
            # global_step이 t보다 커질수록 x가 증가 -> sigmoid 증가 -> alpha 감소
            # x 계산: 변화 속도 조절 (숫자 5는 기울기, 필요시 조절)
            x = 5 * (global_step - t) / t
            sigmoid_x = 1 / (1 + math.exp(-x))

            # sigmoid_x는 0.5부터 시작해서 1.0으로 감
            # 이를 0.0 ~ 1.0 비율로 정규화하여 사용
            decay_ratio = (sigmoid_x - 0.5) * 2
            if decay_ratio > 1.0: decay_ratio = 1.0

            # base(0.8)에서 target(0.2)으로 서서히 이동
            alpha = base_alpha - (base_alpha - target_alpha) * decay_ratio
            # # (혹은 간단하게 step > t 일 때 바로 target_alpha로 고정해도 효과는 봅니다)
            # if alpha < target_alpha: alpha = target_alpha

        # 학습 데이터 gpu에 올리기
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        labels = labels.to(device)

        # 2. Teacher (Original E5) Forward
        with torch.no_grad(): # 원본 모델에게 물어봅니다. "너라면 이거 어디에 배치할래?"
            teacher_outputs = teacher_model(**batch_inputs)

            # teacher_embeddings = teacher_outputs.last_hidden_state.mean(dim=1)
            hidden = teacher_outputs.last_hidden_state       # (B, L, D)
            mask = batch_inputs['attention_mask'].unsqueeze(-1)  # (B, L, 1)
            teacher_embeddings = (hidden * mask).sum(dim=1) / mask.sum(dim=1)

            teacher_embeddings = F.normalize(teacher_embeddings, p=2, dim=1)

        # 3. Student LoRA Forward
        student_vectors = model(**batch_inputs) # B, 3, D
        """
        model(**batch_inputs) 은 자동으로 다음과 같이 해석됨:
        model(
            input_ids=batch_inputs["input_ids"],
            attention_mask=batch_inputs["attention_mask"]
        )
        """

        # 4. 벡터 쪼개기
        genre_vector = student_vectors[:, 0, :] # Vector 0: 장르
        content_vectors = student_vectors[:, 1:, :] # Vector 1: 내용

        # --- Loss Calculation Start ---
        """
        1. loss_contrastive (카테고리 로스): "같은 카테고리니까 뭉쳐! 다른 카테고리는 저리 가!"
            → 힘의 방향: 공간 왜곡 (Distortion)

        2. loss_distill (보존 로스): "야, 그래도 원래 의미(Semantic)가 '우주'랑 'SF'였잖아. 그 근처에 있어."
            → 힘의 방향: 원래 자리 유지 (Preservation)

        결과적으로 모델은 원래 의미 공간을 최대한 유지하면서, 카테고리 분류가 가능할 정도로만 살짝 이동하게 됨
        """
        # 1. supervised contrastive sum over positives
        loss_cont = noisy_contrastive_loss(genre_vector, labels)
        # genre_vector는 이미 E5MultiVectorHead에서 정규화 했으니 ㄱㅊ
        # student_vector 자체가 이미 길이가 1인 벡터임

        # 1-2. ★추가됨★: 장르 벡터 폭주 방지용 KD (Safety Net)
        # 정규화(Normalize) 후 MSE 계산? 왜냐만 길이가 1인 벡터를 평균내서 l2-norm 구하면 1보다 작아지잖아
        # (1,0), (0,1) -> (0.5, 0), (0, 0.5) -> l2-norm(길이) = 0.707
        genre_norm = F.normalize(genre_vector, p=2, dim=1)
        loss_kd_genre = F.mse_loss(genre_norm, teacher_embeddings)

        # 2. distillation loss (MSE)
        # 카테고리 맞추는 것도 좋은데, 원래 위치랑 너무 달라지진 마
        student_content_mean = content_vectors.mean(dim=1)
        student_content_mean = F.normalize(student_content_mean, p=2, dim=1)
        loss_kd_content = F.mse_loss(student_content_mean, teacher_embeddings)

        # KD Loss 결합 (Alpha 적용)
        # loss_kd_combined = alpha * loss_kd_genre + (1 - alpha) * loss_kd_content
        loss_kd_combined = alpha * loss_kd_genre # + (1 - alpha) * loss_kd_content

        # --- 2. Dynamic Weighting (GradNorm 측정 방식) ---
        # 1. Main Task가 어텐션 레이어를 당기는 힘 측정
        norm_main = calc_grad_norm(loss_cont, model.attention)

        # 2. Sub Task가 어텐션 레이어를 당기는 힘 측정
        norm_sub = calc_grad_norm(loss_kd_combined, model.attention)

        # 3. 비율 계산 (Sub의 힘이 0이면 나눗셈 에러 나니까 1e-8 추가)
        # 목표: norm_sub * ratio가 norm_main과 비슷해지도록 함
        # 즉, ratio = norm_main / norm_sub
        # current_ratio = norm_main / (norm_sub + 1e-8)

        # 3 수정
        # 원래 GradNorm: ratio = Main / Sub  ->  결과: Main 힘 == Sub 힘 (1:1)
        # 수정 GradNorm: ratio = (Main / Sub) * 0.2  -> 결과: Sub 힘이 Main의 20%가 됨 (5:1)
        target_scale = 0.6
        current_ratio = norm_main / (norm_sub + 1e-8) * target_scale

        # [안전장치] 비율이 터무니없이 크면 자름 (Gradient Explosion 방지)
        if current_ratio > 1000.0:
            current_ratio = 1000.0

        # 4. 이동 평균 업데이트 (학습 안정화)
        running_ratio = beta * running_ratio + (1 - beta) * current_ratio

        # [선택] KD Loss가 너무 흔들리면 balance_ratio에 상한선을 둘 수도 있습니다 (예: clamp)
        # balance_ratio = torch.clamp(balance_ratio, max=10000)

        # 최종 Loss: 이제 두 항의 스케일(Magnitude)이 1:1로 맞춰집니다.
        total_loss = loss_cont + running_ratio * loss_kd_combined




        # """++++++++++++++++++++++++++++++++++++++++디버깅++++++++++++++++++++++++++++++++++++++++"""
        # # real_lambda_genre = LAMBDA * alpha
        # # real_lambda_content = LAMBDA * (1 - alpha)
        # real_lambda_genre = running_ratio * alpha
        # real_lambda_content = running_ratio * (1 - alpha)
        # if step % 100 == 0:
        #     # 1. 진단 함수 호출 (딕셔너리 리턴)
        #     # 주의: .item()이 아니라 Tensor 상태의 loss 변수를 넣어야 합니다!
        #     diag_results = compute_multivector_diagnostics(
        #         model,
        #         l_cont=loss_cont,
        #         l_kd_genre=0,
        #         l_kd_content=loss_kd_content,
        #         lambda_genre=real_lambda_genre,
        #         lambda_content=real_lambda_content
        #     )
        #     # A. 힘의 크기 (Norm)
        #     grad_logs["force_cluster"].append(diag_results["Force(Cluster)"])
        #     grad_logs["force_drag"].append(diag_results["Force(Drag)"])
        #     grad_logs["force_anchor"].append(diag_results["Force(Anchor)"])

        #     # B. 힘의 방향 (Cosine)
        #     grad_logs["cos_friction"].append(diag_results["Cos(Friction)"])
        #     grad_logs["cos_conflict"].append(diag_results["Cos(Conflict)"])

        #     grad_logs["step"].append(step + epoch * len(train_loader))
        # """++++++++++++++++++++++++++++++++++++++++디버깅++++++++++++++++++++++++++++++++++++++++"""


        # 역전파 과정
        # total_loss.backward()  # 기울기(Gradient) 계산 완료
        # optimizer.zero_grad()  # 기울기를 0으로 초기화 (삭제) 😱
        # optimizer.step()       # 0이 된 기울기로 가중치 업데이트 (변화 없음)
        optimizer.zero_grad()  # 1. 이전 기울기 청소 이거 맨 위로 옮김
        total_loss.backward()  # 2. 현재 기울기 계산
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)    # 테스트임!!!!
        optimizer.step()       # 3. 업데이트

        scheduler.step() # 스케줄러가 step 단위라면 여기, epoch 단위면 loop 밖으로

        total_train_loss += total_loss.item()
        genre_loss_cont += loss_cont.item()                   # 디버깅
        genre_loss_kd += loss_kd_genre.item()                 # 디버깅
        content_loss_kd += loss_kd_content.item()             # 디버깅
        epoch_loss_sum += total_loss.item()                   # 디버깅
        epoch_norm_main_sum += norm_main                      # 디버깅
        epoch_norm_sub_sum += norm_sub                        # 디버깅
        epoch_ratio_sum += running_ratio                      # 디버깅

    train_loss = total_train_loss / len(train_loader)
    train_genre_loss_cont = genre_loss_cont / len(train_loader)     # 디버깅
    train_genre_loss_kd = genre_loss_kd / len(train_loader)         # 디버깅
    train_content_loss_kd = content_loss_kd / len(train_loader)     # 디버깅
    avg_loss = epoch_loss_sum / len(train_loader)                   # 디버깅
    avg_norm_main = epoch_norm_main_sum / len(train_loader)         # 디버깅
    avg_norm_sub = epoch_norm_sub_sum / len(train_loader)           # 디버깅
    avg_ratio = epoch_ratio_sum / len(train_loader)                 # 디버깅

    model.eval()
    metrics = compute_retrieval_accuracy(model, valid_loader, device, tokenizer)
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "genre_loss_cont": train_genre_loss_cont,
        "genre_loss_kd": train_genre_loss_kd,
        "content_loss_kd": train_content_loss_kd,
        "train/grad_ratio_avg": avg_ratio,      # 이번 에포크 평균 비율
        "train/norm_main_avg": avg_norm_main,   # 메인 태스크 그래디언트 평균 크기
        "train/norm_sub_avg": avg_norm_sub,     # 서브 태스크 그래디언트 평균 크기
        "valid/top1_acc": metrics["top1_acc"],
        "valid/precision@10": metrics["precision@k"],
        "valid/mrr": metrics["mrr"],
        "lr": scheduler.get_last_lr()[0],
    })
    print(f"[Epoch {epoch + 1}] Train Loss: {train_loss:.4f}")
    print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
    print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR              : {metrics['mrr']:.4f}")

Epoch: 1: 100%|████████████████████████████████████████████████████████████████████████████████████| 512/512 [02:23<00:00,  3.56it/s]


[Epoch 1] Train Loss: 2.4149
Top-1 Accuracy : 0.5714 | Top-10 Accuracy : 0.8591
Precision@10   : 0.5328 | MRR              : 0.6701


Epoch: 2: 100%|████████████████████████████████████████████████████████████████████████████████████| 512/512 [02:23<00:00,  3.56it/s]


[Epoch 2] Train Loss: 2.1395
Top-1 Accuracy : 0.5831 | Top-10 Accuracy : 0.8646
Precision@10   : 0.5594 | MRR              : 0.6812


Epoch: 3: 100%|████████████████████████████████████████████████████████████████████████████████████| 512/512 [02:23<00:00,  3.56it/s]


...

[Epoch 18] Train Loss: 1.4700
Top-1 Accuracy : 0.6168 | Top-10 Accuracy : 0.8490
Precision@10   : 0.6061 | MRR              : 0.6994


Epoch: 19: 100%|███████████████████████████████████████████████████████████████████████████████████| 512/512 [02:23<00:00,  3.56it/s]


[Epoch 19] Train Loss: 1.4588
Top-1 Accuracy : 0.6166 | Top-10 Accuracy : 0.8495
Precision@10   : 0.6063 | MRR              : 0.6995


Epoch: 20: 100%|███████████████████████████████████████████████████████████████████████████████████| 512/512 [02:23<00:00,  3.56it/s]


[Epoch 20] Train Loss: 1.4519
Top-1 Accuracy : 0.6159 | Top-10 Accuracy : 0.8497
Precision@10   : 0.6065 | MRR              : 0.6991


In [ ]:
import os
save_path = f"./old_multi_vec_{LAMBDA}_20ep.pth"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(model.state_dict(), save_path)